In [1]:
pwd


'/home/pavani/datascience_e_2_e/datascience_project/research'

In [2]:
import os
os.chdir("../")

In [3]:
pwd


'/home/pavani/datascience_e_2_e/datascience_project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_dir: Path
    unzip_dir: Path

In [5]:
from src.datascience.constants import CONFIG_FILE_PATH,PARAMS_FILE_PATH,SCHEMA_FILE_PATH
from src.datascience.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                                  ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL = config.source_URL,
            local_data_dir = config.local_data_dir,
            unzip_dir = config.unzip_dir)

        return data_ingestion_config

In [7]:
## component- Data Ingestion
import urllib.request as request
import os
from src.datascience import logger
import zipfile
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    ## downloading the zipfile
    def download_file(self):
        if not os.path.exists(self.config.local_data_dir):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_dir
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")


    ##unzipping the downloaded file
    def extract_zip_file(self):
        """
        zip_file_path: str
        Etracts the zip filw into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_dir, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            


In [8]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e



[2026-05-16 09:09:07,154: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-16 09:09:07,159: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-16 09:09:07,161: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-16 09:09:07,162: INFO: common: created directory at: artifacts]
[2026-05-16 09:09:07,164: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-16 09:09:07,165: INFO: 2428785376: File already exists]
